# Run Flask Webapp + ngrok on Kaggle

Notebook ini untuk menjalankan webapp Flask dari project ini di Kaggle dan membuat URL publik menggunakan ngrok.

# Download and Install all requirements

In [1]:
# 1) Clone repository (jika belum ada)
# Ganti URL repo sesuai kebutuhan.
import os
from pathlib import Path

REPO_URL = "https://github.com/Herutriana44/Acute-ischemic-stroke-dataset-image-segmentation.git"
PROJECT_DIR = Path("/content/Acute-ischemic-stroke-dataset-image-segmentation")

if not PROJECT_DIR.exists():
    !git clone "$REPO_URL" "$PROJECT_DIR"

os.chdir(PROJECT_DIR)
print("Working dir:", Path.cwd())

Cloning into '/content/Acute-ischemic-stroke-dataset-image-segmentation'...
remote: Enumerating objects: 466, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 466 (delta 34), reused 43 (delta 21), pack-reused 402 (from 2)
Receiving objects: 100% (466/466), 195.45 MiB | 34.42 MiB/s, done.
Resolving deltas: 100% (220/220), done.
Updating files: 100% (114/114), done.
Working dir: /content/Acute-ischemic-stroke-dataset-image-segmentation


In [2]:
# switch branch "gemini_termux"
# !git switch gemini_termux

In [3]:
# Frontend (Vite)
# - npm install: wajib (membuat node_modules)
# - npm run build: optional (static bundle untuk Flask)
# - npm run dev: serve JS (dev server) di port 5173
%cd webapp/frontend
!npm install

# (Opsional) build production bundle ke webapp/static/dist
!npm run build

%cd ../..

'\n%cd webapp/frontend\n!npm install\n\n# (Opsional) build production bundle ke webapp/static/dist\n!npm run build\n\n%cd ../..\n'

In [4]:
# 2) Install dependencies
!pip install -q -r requirements.txt pyngrok waitress

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 11.0 MB/s eta 0:00:00


In [5]:
# 3) Set ngrok auth token
# Opsi A (direkomendasikan): isi token manual di bawah.
# Opsi B: set ENV di Kaggle Secrets/Environment sebagai NGROK_AUTHTOKEN.

import os
from pyngrok import ngrok

NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN", "1TfbVOS48SeXdQ7rJ2do5JjJFxG_4d5K3jMerctfbUsXvidrT")
# Contoh: NGROK_AUTHTOKEN = "<isi_token_ngrok_anda>"

if not NGROK_AUTHTOKEN:
    raise ValueError("NGROK_AUTHTOKEN belum diset. Isi variabel atau set env NGROK_AUTHTOKEN.")

ngrok.set_auth_token(NGROK_AUTHTOKEN)
print("ngrok token configured")

ngrok token configured


In [6]:
# 4) Jalankan Flask app di background (waitress)
import threading
from waitress import serve

from webapp.app import app

PORT = 5000

def run_app():
    # threads tinggi agar inference upload tetap responsif
    serve(app, host="0.0.0.0", port=PORT, threads=8)

server_thread = threading.Thread(target=run_app, daemon=True)
server_thread.start()

print(f"Flask app started on http://0.0.0.0:{PORT}")

Flask app started on http://0.0.0.0:5000


In [7]:
# 5) Buat public URL ngrok (port forwarding)
from pyngrok import ngrok

# Tutup tunnel lama jika ada
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_tunnel = ngrok.connect(addr=PORT, proto="http")
print("Public URL:", public_tunnel.public_url)

Public URL: https://a30b-136-118-60-130.ngrok-free.app


In [8]:
# 5) Jalankan Vite dev server (serve JS) + buat public URL ngrok untuk Flask & Vite
import os
import subprocess
import time
from pathlib import Path
from pyngrok import ngrok

FLASK_PORT = PORT          # 5000
VITE_PORT = 5173

# Start Vite dev server in background
frontend_dir = Path("webapp/frontend").resolve()
cmd = ["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", str(VITE_PORT), "--strictPort"]

# Hindari double-run kalau cell dieksekusi ulang
if "_vite_proc" not in globals() or (_vite_proc and _vite_proc.poll() is not None):
    _vite_proc = subprocess.Popen(cmd, cwd=str(frontend_dir), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(2)
    print(f"Vite dev server starting on http://0.0.0.0:{VITE_PORT}")
else:
    print("Vite dev server already running")

# Close existing tunnels to avoid confusion
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

# Create tunnels
public_flask = ngrok.connect(addr=FLASK_PORT, proto="http")
public_vite = ngrok.connect(addr=VITE_PORT, proto="http")

print("Public Flask URL:", public_flask.public_url)
print("Public Vite  URL:", public_vite.public_url)

print("\nCatatan:")
print("- Buka Public Flask URL untuk pakai webapp.")
print("- Public Vite URL dipakai untuk serve asset JS jika kamu ingin load dari dev server.")
print("  (Kalau template Flask masih load dari /static, Vite URL ini hanya untuk dev/test.)")

'\nimport os\nimport subprocess\nimport time\nfrom pathlib import Path\nfrom pyngrok import ngrok\n\nFLASK_PORT = PORT          # 5000\nVITE_PORT = 5173\n\n# Start Vite dev server in background\nfrontend_dir = Path("webapp/frontend").resolve()\ncmd = ["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", str(VITE_PORT), "--strictPort"]\n\n# Hindari double-run kalau cell dieksekusi ulang\nif "_vite_proc" not in globals() or (_vite_proc and _vite_proc.poll() is not None):\n    _vite_proc = subprocess.Popen(cmd, cwd=str(frontend_dir), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n    time.sleep(2)\n    print(f"Vite dev server starting on http://0.0.0.0:{VITE_PORT}")\nelse:\n    print("Vite dev server already running")\n\n# Close existing tunnels to avoid confusion\nfor t in ngrok.get_tunnels():\n    ngrok.disconnect(t.public_url)\n\n# Create tunnels\npublic_flask = ngrok.connect(addr=FLASK_PORT, proto="http")\npublic_vite = ngrok.connect(addr=VITE_PORT, proto="http")\n

# Link yang diklik

In [9]:
# 6) (Opsional) monitoring tunnel
from pyngrok import ngrok

print("Active tunnels:")
for t in ngrok.get_tunnels():
    print(f"- {t.public_url} -> {t.config.get('addr')}")

Active tunnels:
- https://a30b-136-118-60-130.ngrok-free.app -> http://localhost:5000
